In [ ]:
import pandas as pd
import numpy as np
import importlib

In [17]:
from pathlib import Path
import os
import sys
import pandas as pd
import importlib

# Proje klasörüne geç
os.chdir(r"C:\Users\sametucak\Bibliometric-Normalization-System (BNS)")

PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INPUT_DIR = PROJECT_ROOT / "input"
OUTPUT_DIR = PROJECT_ROOT / "output"

print(PROJECT_ROOT)

C:\Users\sametucak\Bibliometric-Normalization-System (BNS)


In [ ]:
df = pd.read_excel("input/combining yazar listesi.xlsx")

In [ ]:
df = df.drop(columns=["Unnamed: 0"])

In [ ]:
df.columns = [
    "author_full_name",
    "title",
    "journal",
    "year",
    "times_cited",
    "abstract",
    "keywords",
    "affiliation",
    "author"
]

In [ ]:
def clean_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    text = text.strip()

    text = re.sub(r"\s+", " ", text)

    return text

In [ ]:
for col in df.columns:

    if df[col].dtype == "object":

        df[col] = df[col].apply(clean_text)

In [ ]:
df.isnull().sum()

In [ ]:
df["author_full_name"].nunique()

In [ ]:
df["author_full_name"].value_counts().head(20)

In [ ]:
def normalize_author_name(name):

    if pd.isna(name):
        return ""

    name = str(name)

    # Baştaki ve sondaki boşlukları sil
    name = name.strip()

    # Birden fazla boşluğu teke indir
    name = re.sub(r"\s+", " ", name)

    # Büyük-küçük harfleri düzelt
    name = name.title()

    # Virgülden önce ve sonra boşlukları düzelt
    name = re.sub(r"\s*,\s*", ", ", name)

    return name

In [ ]:
df["normalized_author"] = df["author_full_name"].apply(normalize_author_name)

In [ ]:
df[
    ["author_full_name",
     "normalized_author"]
].head(20)

In [ ]:
df["normalized_author"].nunique()

In [ ]:
import pandas as pd

def split_author(author):

    if pd.isna(author):
        return "", ""

    author = str(author).strip()

    if author == "":
        return "", ""

    if "," in author:

        parts = author.split(",", 1)

        last = parts[0].strip()

        first = parts[1].strip() if len(parts) > 1 else ""

        return last, first

    parts = author.split()

    if len(parts) == 0:
        return "", ""

    if len(parts) == 1:
        return parts[0], ""

    return parts[0], " ".join(parts[1:])

In [ ]:
split_author("Lugli, Gabriele Andrea")

In [ ]:
split_author("Chen Yang")

In [ ]:
def first_initial(first_name):

    if first_name == "":

        return ""

    return first_name[0]

In [ ]:
first_initial("Gabriele Andrea")

In [ ]:
def compare_authors(a1, a2):

    last1, first1 = split_author(a1)
    last2, first2 = split_author(a2)

    if last1 != last2:

        return False

    if first_initial(first1) != first_initial(first2):

        return False

    return True

In [ ]:
compare_authors(
    "Lugli, Gabriele Andrea",
    "Lugli, Gabriele A."
)

In [ ]:
compare_authors(
    "Smith, John",
    "Chen, Yang"
)

In [ ]:
def normalize_first_name(name):

    name = name.lower()

    name = name.replace(".", "")

    name = name.strip()

    return name

In [ ]:
normalize_first_name("Gabriele A.")

In [ ]:
from rapidfuzz import fuzz

def author_score(author1, author2):

    last1, first1 = split_author(author1)
    last2, first2 = split_author(author2)

    score = 0

    # Soyadı
    if last1.lower() == last2.lower():
        score += 40

    # İlk isim benzerliği
    score += fuzz.ratio(
        normalize_first_name(first1),
        normalize_first_name(first2)
    ) * 0.6

    return round(score,2)

In [ ]:
author_score(
    "Lugli, Gabriele Andrea",
    "Lugli, Gabriele A."
)

In [ ]:
author_score(
    "Chen, Yang",
    "Chen, Y."
)

In [ ]:
author_score(
    "Smith, John",
    "Chen, Yang"
)

In [ ]:
df["last_name"] = df["normalized_author"].apply(
    lambda x: split_author(x)[0]
)

In [ ]:
df["normalized_author"].isna().sum()

In [ ]:
(df["normalized_author"] == "").sum()

In [ ]:
df[df["normalized_author"] == ""]

In [ ]:
df = df[df["normalized_author"] != ""].copy()

In [ ]:
df.shape

In [ ]:
df["last_name"] = df["normalized_author"].apply(
    lambda x: split_author(x)[0]
)

In [ ]:
def split_author(author):

    if pd.isna(author):
        return "", ""

    author = str(author).strip()

    if author == "":
        return "", ""

    if "," in author:
        parts = author.split(",", 1)

        last = parts[0].strip()

        first = parts[1].strip() if len(parts) > 1 else ""

        return last, first

    parts = author.split()

    if len(parts) == 0:
        return "", ""

    if len(parts) == 1:
        return parts[0], ""

    return parts[0], " ".join(parts[1:])

In [ ]:
df = df[df["normalized_author"] != ""].copy()

In [ ]:
df.shape

In [ ]:
df["last_name"] = df["normalized_author"].apply(
    lambda x: split_author(x)[0]
)

In [ ]:
last_name_groups = df.groupby("last_name")

In [ ]:
# =====================================================
# AUTHOR NORMALIZATION
# =====================================================

# Bibliometric Normalization System (BNS)

## Module 03 - Author Normalization

**Objective**

This notebook identifies and merges author name variants in the Web of Science dataset.

Outputs:

- Merged_Author_Table.xlsx
- Normalized_WoS_Data.xlsx

In [ ]:
author_groups = {}

for last_name, group in df.groupby("last_name"):
    author_groups[last_name] = (
        group["normalized_author"]
        .dropna()
        .unique()
        .tolist()
    )

In [ ]:
df["last_name"] = df["normalized_author"].apply(
    lambda x: split_author(x)[0]
)

In [ ]:
from rapidfuzz import fuzz

def author_name_similarity(author1, author2):

    author1 = str(author1).strip().lower()
    author2 = str(author2).strip().lower()

    return fuzz.token_sort_ratio(author1, author2)

In [ ]:
author_name_similarity(
    "Lugli, Gabriele Andrea",
    "Lugli, Gabriele A."
)

In [ ]:
def find_candidates(author_list, threshold=90):

    candidates = []

    for i in range(len(author_list)):

        for j in range(i + 1, len(author_list)):

            score = author_name_similarity(
                author_list[i],
                author_list[j]
            )

            if score >= threshold:

                candidates.append({
                    "author_1": author_list[i],
                    "author_2": author_list[j],
                    "score": score
                })

    return candidates

In [ ]:
first_lastname = list(author_groups.keys())[0]

print(first_lastname)

In [ ]:
candidate_pairs = find_candidates(
    author_groups[first_lastname]
)

candidate_pairs

In [ ]:
largest_group = max(
    author_groups.items(),
    key=lambda x: len(x[1])
)

largest_lastname = largest_group[0]

print(largest_lastname)
print(len(largest_group[1]))

## 5.1 Affiliation Standardization

In [ ]:
def normalize_affiliation(aff):

    if pd.isna(aff):
        return ""

    aff = str(aff).lower()

    aff = aff.replace("univ.", "university")
    aff = aff.replace("dept.", "department")
    aff = aff.replace("&", "and")

    aff = " ".join(aff.split())

    return aff

In [ ]:
df["normalized_affiliation"] = (
    df["affiliation"]
    .apply(normalize_affiliation)
)

In [ ]:
df[[
    "affiliation",
    "normalized_affiliation"
]].head()

## 5.2 Keyword Standardization

In [ ]:
def normalize_keywords(text):

    if pd.isna(text):
        return ""

    text = str(text).lower()

    text = text.replace(";", ",")

    keywords = [
        x.strip()
        for x in text.split(",")
        if x.strip()
    ]

    keywords = sorted(set(keywords))

    return ",".join(keywords)

In [ ]:
df["normalized_keywords"] = (
    df["keywords"]
    .apply(normalize_keywords)
)

In [ ]:
df[[
    "keywords",
    "normalized_keywords"
]].head()

BÖLÜM 3 — Publication Year

In [ ]:
df["year"] = pd.to_numeric(
    df["year"],
    errors="coerce"
)

In [ ]:
df["year"].describe()

BÖLÜM 4 — Name Similarity

In [ ]:
from rapidfuzz import fuzz

def name_similarity(a, b):

    return fuzz.token_sort_ratio(
        str(a),
        str(b)
    )

BÖLÜM 5 — Affiliation Similarity

In [ ]:
def affiliation_similarity(a, b):

    return fuzz.token_sort_ratio(
        str(a),
        str(b)
    )

BÖLÜM 6 — Keyword Similarity

In [ ]:
def keyword_similarity(a, b):

    return fuzz.token_sort_ratio(
        str(a),
        str(b)
    )

BÖLÜM 7 — Year Similarity

In [ ]:
def year_similarity(y1, y2):

    if pd.isna(y1) or pd.isna(y2):
        return 50

    diff = abs(y1 - y2)

    if diff == 0:
        return 100

    if diff <= 2:
        return 80

    if diff <= 5:
        return 60

    return 20

## 5.3 Author Confidence Score

In [ ]:
def confidence_score(
    author1,
    author2,
    aff1,
    aff2,
    key1,
    key2,
    year1,
    year2
):

    name_score = name_similarity(author1, author2)

    aff_score = affiliation_similarity(aff1, aff2)

    keyword_score = keyword_similarity(key1, key2)

    year_score = year_similarity(year1, year2)

    final_score = (
        name_score * 0.50
        + aff_score * 0.25
        + keyword_score * 0.15
        + year_score * 0.10
    )

    return {
        "name": round(name_score,2),
        "affiliation": round(aff_score,2),
        "keywords": round(keyword_score,2),
        "year": round(year_score,2),
        "confidence": round(final_score,2)
    }

In [ ]:
confidence_score(

    "Lugli, Gabriele Andrea",
    "Lugli, Gabriele A.",

    "University of Parma",
    "Univ. of Parma",

    "microbiome, probiotics",
    "probiotics, microbiome",

    2022,
    2023
)

In [ ]:
df.head(10)

In [ ]:
row1 = df.iloc[0]
row2 = df.iloc[1]

In [ ]:
confidence_score(

    row1["normalized_author"],
    row2["normalized_author"],

    row1["normalized_affiliation"],
    row2["normalized_affiliation"],

    row1["normalized_keywords"],
    row2["normalized_keywords"],

    row1["year"],
    row2["year"]

)

In [ ]:
def evaluate_pair(row1,row2):

    score = confidence_score(

        row1["normalized_author"],
        row2["normalized_author"],

        row1["normalized_affiliation"],
        row2["normalized_affiliation"],

        row1["normalized_keywords"],
        row2["normalized_keywords"],

        row1["year"],
        row2["year"]

    )

    return score

In [ ]:
evaluate_pair(
    df.iloc[0],
    df.iloc[1]
)

In [ ]:
def merge_decision(score):

    if score["confidence"] >= 95:
        return "Automatic Merge"

    elif score["confidence"] >= 85:
        return "Manual Review"

    else:
        return "Different Authors"

In [ ]:
score = evaluate_pair(
    df.iloc[0],
    df.iloc[1]
)

merge_decision(score)

# 6. Author Merge Engine

In [ ]:
merge_table = []

In [ ]:
for lastname in author_groups:

    print(lastname)

In [ ]:
for lastname in author_groups:

    authors = author_groups[lastname]

    print(lastname, len(authors))

In [ ]:
for lastname in author_groups:

    authors = author_groups[lastname]

    if len(authors) < 2:
        continue

    print(lastname)

In [ ]:
merge_table = []

for lastname in author_groups:

    authors = author_groups[lastname]

    if len(authors) < 2:
        continue

    for i in range(len(authors)):

        for j in range(i+1, len(authors)):

            merge_table.append({

                "lastname": lastname,

                "author1": authors[i],

                "author2": authors[j]

            })

In [ ]:
len(merge_df)

In [ ]:
author_lookup = (
    df
    .drop_duplicates("normalized_author")
    .set_index("normalized_author", drop=False)
)

In [ ]:
author_lookup.head()

In [ ]:
scores = []

for _, row in merge_df.iterrows():

    try:

        a = author_lookup.loc[row["author1"]]
        b = author_lookup.loc[row["author2"]]

        result = confidence_score(

            a["normalized_author"],
            b["normalized_author"],

            a["normalized_affiliation"],
            b["normalized_affiliation"],

            a["normalized_keywords"],
            b["normalized_keywords"],

            a["year"],
            b["year"]

        )

        scores.append(result["confidence"])

    except Exception as e:

        print("HATA:")
        print(row["author1"])
        print(row["author2"])
        print(e)

        break

In [ ]:
drop=False

In [ ]:
scores = []

for _, row in merge_df.iterrows():

    a = author_lookup.loc[row["author1"]]
    b = author_lookup.loc[row["author2"]]

    result = confidence_score(

        a["normalized_author"],
        b["normalized_author"],

        a["normalized_affiliation"],
        b["normalized_affiliation"],

        a["normalized_keywords"],
        b["normalized_keywords"],

        a["year"],
        b["year"]

    )

    scores.append(result["confidence"])

# 7. Canonical Author Selection

In [ ]:
automatic_merge = merge_df[
    merge_df["decision"] == "Automatic Merge"
].copy()

automatic_merge.head()

In [ ]:
len(automatic_merge)

In [ ]:
def choose_canonical_name(author1, author2):

    # Daha uzun isim tercih edilir
    if len(author1) >= len(author2):
        return author1
    else:
        return author2

In [ ]:
automatic_merge["canonical_author"] = automatic_merge.apply(
    lambda row: choose_canonical_name(
        row["author1"],
        row["author2"]
    ),
    axis=1
)

In [ ]:
automatic_merge[
    ["author1", "author2", "canonical_author", "confidence"]
].head(20)

In [ ]:
merge_df.columns.tolist()

In [ ]:
author_lookup.head()

In [ ]:
author_lookup.columns.tolist()

In [ ]:
row = merge_df.iloc[0]

row

In [ ]:
a = author_lookup.loc[row["author1"]]
b = author_lookup.loc[row["author2"]]

print(a["normalized_author"])
print(b["normalized_author"])

In [ ]:
confidence_score(

    a["normalized_author"],
    b["normalized_author"],

    a["normalized_affiliation"],
    b["normalized_affiliation"],

    a["normalized_keywords"],
    b["normalized_keywords"],

    a["year"],
    b["year"]
)

# =====================================================
# Bibliometric Normalization System (BNS)
# Module 04 - Merge Engine
# Version : 1.0
# Author  : <Samet UÇAK>
# =====================================================

# Bibliometric Normalization System (BNS)

## Module 04 — Merge Engine

Bu modül aynı araştırmacıya ait isim varyasyonlarını tespit eder, güven puanını hesaplar ve Canonical Author tablosunu oluşturur.

In [ ]:
print("="*70)
print("BIBLIOMETRIC NORMALIZATION SYSTEM")
print("MODULE 04 - MERGE ENGINE")
print("="*70)

In [ ]:
required_columns = [
    "normalized_author",
    "last_name",
    "normalized_affiliation",
    "normalized_keywords",
    "year"
]

missing = [c for c in required_columns if c not in df.columns]

if missing:
    print("Eksik sütunlar:")
    print(missing)
else:
    print("✓ Tüm gerekli sütunlar mevcut.")

In [ ]:
author_lookup = (
    df
    .drop_duplicates("normalized_author")
    .set_index("normalized_author", drop=False)
)

In [ ]:
print(author_lookup.shape)

In [ ]:
merge_table = []

for lastname, authors in author_groups.items():

    if len(authors) < 2:
        continue

    for i in range(len(authors)):

        for j in range(i+1, len(authors)):

            merge_table.append({

                "lastname": lastname,

                "author1": authors[i],

                "author2": authors[j]

            })

In [ ]:
merge_df = pd.DataFrame(merge_table)

print(merge_df.shape)
merge_df.head()

In [ ]:
merge_df["confidence"] = merge_df.apply(
    calculate_confidence,
    axis=1
)

In [ ]:
calculate_confidence

In [ ]:
def calculate_confidence(row):

    a = author_lookup.loc[row["author1"]]
    b = author_lookup.loc[row["author2"]]

    result = confidence_score(

        a["normalized_author"],
        b["normalized_author"],

        a["normalized_affiliation"],
        b["normalized_affiliation"],

        a["normalized_keywords"],
        b["normalized_keywords"],

        a["year"],
        b["year"]

    )

    return result["confidence"]

In [ ]:
calculate_confidence(merge_df.iloc[0])

In [ ]:
merge_df["confidence"] = merge_df.apply(
    calculate_confidence,
    axis=1
)

In [ ]:
print("confidence_score:", callable(confidence_score))
print("merge_decision:", callable(merge_decision))

In [ ]:
def calculate_confidence(row):

    author1 = row["author1"]
    author2 = row["author2"]

    a = author_lookup.loc[author1]
    b = author_lookup.loc[author2]

    score = confidence_score(

        a["normalized_author"],
        b["normalized_author"],

        a["normalized_affiliation"],
        b["normalized_affiliation"],

        a["normalized_keywords"],
        b["normalized_keywords"],

        a["year"],
        b["year"]

    )

    return score["confidence"]

In [ ]:
test_row = merge_df.iloc[0]

calculate_confidence(test_row)

In [ ]:
merge_df["confidence"] = merge_df.apply(
    calculate_confidence,
    axis=1
)

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [ ]:
print("✓ similarity.py başarıyla yüklendi.")

In [ ]:
confidence_score(

    "Lugli, Gabriele Andrea",

    "Lugli, Gabriele A.",

    "University of Parma",

    "University of Parma",

    "microbiome, probiotics",

    "probiotics, microbiome",

    2022,

    2023

)

In [ ]:
from src.merge_engine import (
    build_merge_candidates,
    score_merge_candidates
)

In [ ]:
merge_df = build_merge_candidates(author_groups)

merge_df.head()

In [ ]:
merge_df.shape

In [ ]:
merge_df = score_merge_candidates(

    merge_df,

    author_lookup

)

In [ ]:
merge_df.head()

In [ ]:
from src.merge_engine import (
    build_merge_candidates,
    score_merge_candidates
)

In [ ]:
from src.merge_engine import select_canonical_author

MODÜL 04 – BÖLÜM 2: Canonical Author Mapping

In [ ]:
automatic_merge = merge_df[
    merge_df["decision"] == "Automatic Merge"
].copy()

print(f"Otomatik birleşecek kayıt sayısı: {len(automatic_merge)}")
automatic_merge.head()

In [ ]:
print(df.shape)

In [ ]:
print(author_groups is not None)

In [ ]:
print(author_lookup.shape)

In [ ]:
from src.merge_engine import (
    build_merge_candidates,
    score_merge_candidates
)

merge_df = build_merge_candidates(author_groups)

print(merge_df.shape)
merge_df.head()

In [ ]:
print(df.columns.tolist())

In [ ]:
author_lookup = (
    df
    .sort_values("times_cited", ascending=False)
    .drop_duplicates(subset="normalized_author")
    .set_index("normalized_author")
)

print(author_lookup.shape)
author_lookup.head()

In [ ]:
merge_df = build_merge_candidates(author_groups)

merge_df = score_merge_candidates(
    merge_df,
    author_lookup
)

merge_df.head()

In [ ]:
import src.merge_engine as me

print(me.__file__)

In [ ]:
import src.similarity as sim

print(dir(sim))

In [ ]:
import src.merge_engine as me

print(me.__file__)
print("confidence_score" in dir(me))
print(dir(me))

In [ ]:
import src.merge_engine as me

print(me.confidence_score)

In [ ]:
importlib.reload(sim)
importlib.reload(me)

print("✓ similarity.py yüklendi")
print("✓ merge_engine.py yüklendi")

In [ ]:
import os

print(os.getcwd())

In [ ]:
import pandas as pd
import importlib

import src.similarity as sim
import src.merge_engine as me

print("Modüller başarıyla yüklendi.")

In [ ]:
from pathlib import Path

data_path = Path("../data")

print("Data klasöründeki dosyalar:\n")

for file in data_path.iterdir():
    print(file.name)

In [ ]:
from pathlib import Path

print("Proje klasöründeki dosya ve klasörler:\n")

for item in Path("..").iterdir():
    print(item.name)

In [ ]:
from pathlib import Path

INPUT_DIR = Path("..") / "input"

print("Input klasöründeki dosyalar:\n")

for file in INPUT_DIR.iterdir():
    print(file.name)

In [ ]:
from pathlib import Path

OUTPUT_DIR = PROJECT_ROOT / "output"

df = pd.read_excel(
    OUTPUT_DIR / "Normalized_WoS_Data.xlsx"
)

print(df.shape)
print(df.columns.tolist())

In [ ]:
print(df.columns.tolist())

In [ ]:
df.head(5)

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("..") / "output"

print("Output klasörü:\n")

for file in OUTPUT_DIR.iterdir():
    print(file.name)

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR = PROJECT_ROOT / "output"

print("Project Root :", PROJECT_ROOT)
print("Input :", INPUT_DIR)
print("Output:", OUTPUT_DIR)

In [ ]:
import pandas as pd
import importlib

import src.merge_engine as me

importlib.reload(me)

print("Merge Engine yüklendi.")

In [ ]:
df = pd.read_excel(
    INPUT_DIR / "Normalized_WoS_Data.xlsx"
)

print(df.shape)
df.head()

In [ ]:
import importlib
import src.merge_engine as me

importlib.reload(me)

print("merge_engine.py yeniden yüklendi.")

In [ ]:
author_lookup = me.build_author_lookup(df)

print(author_lookup.shape)

In [ ]:
print(df["times_cited"].dtype)

print(df["times_cited"].head(20))

In [ ]:
import importlib
import src.merge_engine as me

importlib.reload(me)

print("merge_engine yeniden yüklendi.")

In [ ]:
author_lookup = me.build_author_lookup(df)

print(author_lookup.shape)

In [ ]:
author_groups = me.build_author_groups(df)

print(f"Soyadı grubu sayısı: {len(author_groups)}")

In [ ]:
merge_df = me.build_merge_candidates(author_groups)

print(merge_df.shape)

merge_df.head()

In [ ]:
merge_df = me.score_merge_candidates(
    merge_df,
    author_lookup
)

print(merge_df.shape)

merge_df.head()

In [ ]:
print(author_lookup.columns.tolist())

In [ ]:
print(author_lookup.index.name)

In [ ]:
import importlib
import src.merge_engine as me

importlib.reload(me)

In [ ]:
import src.merge_engine as me

print(me.__file__)

In [ ]:
import inspect

print(inspect.getsource(me.calculate_confidence))

In [ ]:
import importlib
import src.merge_engine as me

me = importlib.reload(me)

print("Reload başarılı")

In [ ]:
print(me.calculate_confidence)
print(me.score_merge_candidates)

In [ ]:
import inspect

print(inspect.getsource(me.score_merge_candidates))

In [ ]:
row = merge_df.iloc[0]

print(row)

In [ ]:
a = author_lookup.loc[row["author1"]]
b = author_lookup.loc[row["author2"]]

print(a)
print()
print(b)

In [ ]:
print(type(a))
print(type(b))

In [ ]:
import src.similarity as sim

score = sim.confidence_score(
    row["author1"],
    row["author2"],
    a["normalized_affiliation"],
    b["normalized_affiliation"],
    a["normalized_keywords"],
    b["normalized_keywords"],
    a["year"],
    b["year"]
)

print(score)

In [ ]:
print(me.calculate_confidence(
    merge_df.iloc[0],
    author_lookup
))

In [ ]:
merge_df.iloc[:5].apply(
    lambda r: me.calculate_confidence(r, author_lookup),
    axis=1
)

In [ ]:
import importlib
import src.merge_engine as me

me = importlib.reload(me)

In [ ]:
merge_df = me.score_merge_candidates(
    merge_df,
    author_lookup
)

In [ ]:
print(merge_df.head())

In [ ]:
merge_df["decision"].value_counts()

In [ ]:
automatic_merge = merge_df[
    merge_df["decision"] == "Automatic Merge"
].copy()

print(automatic_merge.shape)

automatic_merge.head()

In [ ]:
automatic_merge["canonical_author"] = automatic_merge.apply(

    lambda row: me.select_canonical_author(
        row["author1"],
        row["author2"]
    ),

    axis=1

)

automatic_merge.head()

In [ ]:
import importlib
import src.indicator as ind

importlib.reload(ind)

print("indicator.py yüklendi.")

In [ ]:
import os
from pathlib import Path

print("Mevcut dizin:", Path.cwd())

In [ ]:
import os
from pathlib import Path

os.chdir("..")

PROJECT_ROOT = Path.cwd()

print("Proje dizini:", PROJECT_ROOT)
print("src var mı?", (PROJECT_ROOT / "src").exists())
print("__init__.py var mı?", (PROJECT_ROOT / "src" / "__init__.py").exists())

In [ ]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import importlib
import src.indicator as ind

importlib.reload(ind)

print("indicator.py başarıyla yüklendi.")

In [ ]:
from pathlib import Path

src_path = PROJECT_ROOT / "src"

print("src içeriği:\n")

for file in src_path.iterdir():
    print(file.name)

In [ ]:
import importlib
import src.indicator as ind

importlib.reload(ind)

print("indicator.py başarıyla yüklendi.")

In [ ]:
import importlib
import src.indicator as ind

importlib.reload(ind)

print("indicator.py yüklendi.")

In [ ]:
df["times_cited"] = (
    pd.to_numeric(
        df["times_cited"],
        errors="coerce"
    )
    .fillna(0)
)

In [ ]:
import pandas as pd
import numpy as np
import importlib

In [ ]:
df["times_cited"] = (
    pd.to_numeric(
        df["times_cited"],
        errors="coerce"
    )
    .fillna(0)
)

In [ ]:
print(df.columns.tolist())
print(df.shape)

In [ ]:
print(PROJECT_ROOT)
print(type(PROJECT_ROOT))

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(PROJECT_ROOT)

OUTPUT_DIR = PROJECT_ROOT / "output"

df = pd.read_excel(
    OUTPUT_DIR / "Normalized_WoS_Data.xlsx"
)

print(df.shape)
print(df.columns.tolist())

In [ ]:
from pathlib import Path
import os
import sys

# Proje klasörüne git
os.chdir(r"C:\Users\sametucak\Bibliometric-Normalization-System (BNS)")

PROJECT_ROOT = Path.cwd()
INPUT_DIR = PROJECT_ROOT / "input"
OUTPUT_DIR = PROJECT_ROOT / "output"

# src klasörünü Python'a tanıt
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INPUT_DIR var mı?:", INPUT_DIR.exists())
print("OUTPUT_DIR var mı?:", OUTPUT_DIR.exists())
print("src var mı?:", (PROJECT_ROOT / "src").exists())

In [ ]:
import pandas as pd

df = pd.read_excel(OUTPUT_DIR / "Normalized_WoS_Data.xlsx")

print(df.shape)
print(df.columns.tolist())

In [ ]:
df["times_cited"] = (
    pd.to_numeric(df["times_cited"], errors="coerce")
    .fillna(0)
    .astype(int)
)

In [ ]:
print(df["times_cited"].dtype)

In [ ]:
import importlib
import src.indicator as ind

importlib.reload(ind)

pub = ind.publication_count(df)

print(pub.shape)
pub.head()

In [ ]:
cit = ind.citation_count(df)

print(cit.shape)
cit.head()

In [ ]:
import src.indicator as ind

print(dir(ind))

In [ ]:
import importlib
import src.indicator as ind

importlib.reload(ind)

print(dir(ind))

In [ ]:
cit = ind.citation_count(df)

print(cit.shape)

cit.head()

In [ ]:
author_metrics = pub.merge(
    cit,
    on="normalized_author",
    how="left"
)

print(author_metrics.shape)

author_metrics.head()

In [ ]:
import importlib
import src.indicator as ind

importlib.reload(ind)

In [ ]:
avg = ind.average_citations(df)

print(avg.shape)

avg.head()

In [ ]:
import importlib
import src.indicator as ind

importlib.reload(ind)

In [ ]:
h = ind.h_index(df)

print(h.shape)

h.head()

In [ ]:
author_metrics = (
    pub
    .merge(cit, on="normalized_author")
    .merge(avg, on="normalized_author")
    .merge(h, on="normalized_author")
)

print(author_metrics.shape)

author_metrics.head()

In [ ]:
import importlib
import src.indicator as ind

importlib.reload(ind)

In [ ]:
i10 = ind.i10_index(df)

print(i10.shape)

i10.head()

In [ ]:
author_metrics = (
    pub
    .merge(cit, on="normalized_author")
    .merge(avg, on="normalized_author")
    .merge(h, on="normalized_author")
    .merge(i10, on="normalized_author")
)

print(author_metrics.shape)

author_metrics.head()

In [ ]:
import importlib
import src.indicator as ind

importlib.reload(ind)

In [ ]:
first = ind.first_publication_year(df)

last = ind.last_publication_year(df)

age = ind.academic_age(df)

print(first.head())

print(last.head())

print(age.head())

In [ ]:
import importlib
import src.indicator as ind

importlib.reload(ind)

In [ ]:
metrics = ind.build_author_metrics(df)

print(metrics.shape)

metrics.head()

In [ ]:
print(dir(ind))

In [ ]:
import src.indicator as ind

print(ind.__file__)

In [ ]:
import importlib
import src.indicator as ind

ind = importlib.reload(ind)

print(dir(ind))

In [ ]:
metrics = ind.build_author_metrics(df)

print(metrics.shape)

metrics.head()

In [7]:
import importlib
import src.exporter as ex

importlib.reload(ex)

print("Exporter yüklendi.")

Exporter yüklendi.


In [5]:
import importlib
import src.exporter as ex

importlib.reload(ex)

print(ex.__file__)

C:\Users\sametucak\Bibliometric-Normalization-System (BNS)\src\exporter.py


In [9]:
import importlib
import src.exporter as ex

importlib.reload(ex)

print(ex.__file__)

C:\Users\sametucak\Bibliometric-Normalization-System (BNS)\src\exporter.py


In [11]:
ex.export_csv(
    metrics,
    OUTPUT_DIR / "Author_Metrics.csv"
)

NameError: name 'metrics' is not defined

In [13]:
print(df.shape)

NameError: name 'df' is not defined

In [15]:
import pandas as pd

df = pd.read_excel(
    OUTPUT_DIR / "Normalized_WoS_Data.xlsx"
)

df["times_cited"] = (
    pd.to_numeric(df["times_cited"], errors="coerce")
      .fillna(0)
      .astype(int)
)

metrics = ind.build_author_metrics(df)

NameError: name 'OUTPUT_DIR' is not defined

In [19]:
import src.indicator as ind
import src.exporter as ex

importlib.reload(ind)
importlib.reload(ex)

print("Modules loaded.")

Modules loaded.


In [21]:
df = pd.read_excel(
    OUTPUT_DIR / "Normalized_WoS_Data.xlsx"
)

df["times_cited"] = (
    pd.to_numeric(df["times_cited"], errors="coerce")
      .fillna(0)
      .astype(int)
)

print(df.shape)

(12197, 13)


In [23]:
ex.export_author_metrics(
    metrics,
    OUTPUT_DIR
)

AttributeError: module 'src.exporter' has no attribute 'export_author_metrics'

In [25]:
ex.export_csv(
    metrics,
    OUTPUT_DIR / "Author_Metrics.csv"
)

NameError: name 'metrics' is not defined

In [27]:
print(dir(ex))

['Path', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'export_csv', 'export_excel', 'pd']


In [29]:
import importlib
import src.exporter as ex

importlib.reload(ex)

print(dir(ex))

['Path', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'export_all_outputs', 'export_author_metrics', 'export_csv', 'export_excel', 'export_merge_report', 'export_normalized_data', 'pd']


In [31]:
metrics = ind.build_author_metrics(df)

print(metrics.shape)

(10381, 9)


In [32]:
ex.export_author_metrics(
    metrics,
    OUTPUT_DIR
)

Saved: C:\Users\sametucak\Bibliometric-Normalization-System (BNS)\output\Author_Metrics.xlsx


In [35]:
ex.export_csv(
    metrics,
    OUTPUT_DIR / "Author_Metrics.csv"
)

Saved: C:\Users\sametucak\Bibliometric-Normalization-System (BNS)\output\Author_Metrics.csv


In [37]:
tree src

SyntaxError: invalid syntax (3242186997.py, line 1)

In [39]:
ls -R src

 Volume in drive C has no label.
 Volume Serial Number is A272-9579

 Directory of C:\Users\sametucak\Bibliometric-Normalization-System (BNS)


 Directory of C:\Users\sametucak\Bibliometric-Normalization-System (BNS)\src

10.07.2026  10:16    <DIR>          .
10.07.2026  10:16    <DIR>          ..
08.07.2026  17:12    <DIR>          .ipynb_checkpoints
08.07.2026  12:23                 0 __init__.py
10.07.2026  10:01    <DIR>          __pycache__
08.07.2026  15:49             1.815 author_normalization.py
08.07.2026  10:55               198 cleaning.py
10.07.2026  10:16                 0 config.py
10.07.2026  10:16                 0 exceptions.py
10.07.2026  10:06             1.817 exporter.py
08.07.2026  17:14             4.068 indicator.py
08.07.2026  16:15             4.221 merge_engine.py
08.07.2026  11:04             1.742 similarity.py
08.07.2026  10:58               176 utils.py
              10 File(s)         14.037 bytes
               4 Dir(s)  38.449.668.096 bytes free


File Not Found


In [41]:
import pandas as pd

from src.author_normalization import normalize_dataset

In [43]:
df = pd.DataFrame({
    "author": ["Smith, John"],
    "affiliation": ["MIT"]
})

normalize_dataset(df)

ValidationError: Missing required columns: keywords

In [45]:
df = pd.DataFrame({
    "author": ["Smith, John"],
    "affiliation": ["MIT"],
    "keywords": ["AI; Bibliometrics"]
})

normalize_dataset(df)

,author,affiliation,keywords,normalized_author,last_name,normalized_affiliation,normalized_keywords
0,"Smith, John",MIT,AI; Bibliometrics,"Smith, John",Smith,mit,ai; bibliometrics


In [47]:
from src.similarity import confidence_score
from src.similarity import merge_decision

In [49]:
score = confidence_score(
    "Smith, John",
    "Smith, John",
    "MIT",
    "MIT",
    "AI",
    "AI",
    2024,
    2024
)

print(score)
print(merge_decision(score))

100.0
Automatic Merge


In [51]:
import pandas as pd

from src.indicator import build_author_metrics

In [53]:
df = pd.DataFrame({
    "normalized_author": ["Smith, John"],
    "times_cited": [15]
})

build_author_metrics(df)

KeyError: 'Column not found: year'

In [55]:
import importlib
import src.indicator

importlib.reload(src.indicator)

from src.indicator import build_author_metrics

In [57]:
import pandas as pd

df = pd.DataFrame({
    "normalized_author": ["Smith, John"],
    "times_cited": [15]
})

build_author_metrics(df)

ValidationError: Missing required columns: year

In [59]:
import pandas as pd

df = pd.DataFrame({
    "normalized_author": [
        "Smith, John",
        "Smith, John",
        "Brown, Alice"
    ],
    "times_cited": [
        15,
        8,
        22
    ],
    "year": [
        2020,
        2022,
        2021
    ]
})

metrics = build_author_metrics(df)

metrics

,normalized_author,publication_count,citation_count,average_citations,h_index,i10_index,first_publication_year,last_publication_year,academic_age
0,"Brown, Alice",1,22,22.0,1,1,2021,2021,1
1,"Smith, John",2,23,11.5,2,1,2020,2022,3


In [61]:
import pandas as pd

from src.cleaning import clean_data

df = pd.DataFrame({
    " author ": ["Smith", "Smith", None],
    "year": [2020, 2020, None]
})

clean_df = clean_data(df)

clean_df

,author,year
0,Smith,2020.0


In [65]:
python main.py

SyntaxError: invalid syntax (1578409569.py, line 1)

In [67]:
Bibliometric-Normalization-System (BNS)

NameError: name 'Bibliometric' is not defined

In [69]:
python main.py

SyntaxError: invalid syntax (1578409569.py, line 1)